# ML-07 — Baseline Action Score and Top-20 Review

This baseline is a transparent, rule-based comparison point. It is intentionally narrow and is not a learned model.

## 1. Rule and scope

Flag a page for review when it has at least 500 rolling-90-day impressions and has not been updated for at least 180 days. The notebook reports both the rule count and its share of each relevant denominator.

In [ ]:
import os
import pandas as pd

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
df_clean = df_raw.loc[
    (df_raw['impressions_90d'] >= 10) & (df_raw['content_age_days'] >= 90)
] .copy()
high_impression = df_clean[df_clean['impressions_90d'] >= 500]
baseline_queue = high_impression[
    high_impression['days_since_last_update'] >= 180
] .copy()

baseline_summary = pd.DataFrame({
    'Measure': [
        'Active corpus',
        'High-impression pages',
        'Stale high-impression pages',
        'Share of active corpus (%)',
        'Share of high-impression pages (%)',
    ],
    'Value': [
        len(df_clean),
        len(high_impression),
        len(baseline_queue),
        round(len(baseline_queue) / len(df_clean) * 100, 3),
        round(len(baseline_queue) / len(high_impression) * 100, 3),
    ],
})
display(baseline_summary)


## 2. Ranked review queue

The rule does not claim impact. Within the flagged set, pages are ordered by impressions so an editor can begin with the largest observed reach. IDs are pseudonymized.

In [ ]:
baseline_queue = baseline_queue.sort_values('impressions_90d', ascending=False).copy()
baseline_queue['baseline_rank'] = range(1, len(baseline_queue) + 1)
baseline_queue['reason_code'] = 'STALE_HIGH_REACH'

preview_columns = [
    'baseline_rank', 'reason_code', 'impressions_90d', 'avg_position',
    'ctr', 'days_since_last_update', 'engagement_rate',
]
display(baseline_queue[preview_columns].head(20))


## 3. Interpretation and limits

This is a transparent operational screen, not evidence that refreshing a page will improve its outcome. Its narrow coverage is reported from code rather than described with an ambiguous denominator. A human should inspect intent, factual accuracy, redirects, links, and YMYL risk before acting.

## Self-check

- [x] Baseline thresholds and denominators are explicit.
- [x] No result values or verdicts are hard-coded in prose.
- [x] The queue is decision-support only.